# Module 17 Lab — Google ADK 2.x: Hierarchical Agent Trees

In this lab you will build a multi-tier research system using Google ADK's composable agent primitives:

```
Orchestrator (LlmAgent)
├── ResearchPipeline (SequentialAgent)
│   ├── Planner (LlmAgent)
│   ├── ParallelSearch (ParallelAgent)
│   │   ├── SearcherA (LlmAgent)
│   │   └── SearcherB (LlmAgent)
│   └── Synthesiser (LlmAgent)
└── Critic (LlmAgent)
```

**By the end of this lab you will be able to:**
- Define tools as plain Python functions and understand how docstrings become schemas
- Compose `SequentialAgent` and `ParallelAgent` into a processing pipeline
- Use `DatabaseSessionService` to persist session state across runs
- Launch an ADK app locally with `adk dev`

**Prerequisites:** Google AI Studio API key. Get one free at https://aistudio.google.com/app/apikey

**Time estimate:** 45–60 minutes

## Setup — Install Dependencies

In [ ]:
!pip install "google-adk>=2.2.0"

## Environment Setup

In Google Colab, add your API key as a secret named `GOOGLE_API_KEY` (key icon in the left sidebar), then run the cell below.

Locally, set it as an environment variable: `export GOOGLE_API_KEY=your_key_here`

In [ ]:
import os
import asyncio
from google.adk.agents import LlmAgent, SequentialAgent, ParallelAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, DatabaseSessionService
from google.genai.types import Content, Part

# In Colab: load from Colab secrets
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # not in Colab — rely on environment variable

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
if not GOOGLE_API_KEY:
    raise ValueError("Set GOOGLE_API_KEY (Google AI Studio key) in environment")

MODEL = "gemini-2.5-flash"
print(f"Using model: {MODEL}")
print("Environment ready.")

## Section 1 — Defining Tools

In Google ADK 2.x, a tool is a plain Python function — no decorator needed. ADK inspects each function automatically and does three things:

1. **Docstring** — becomes the tool's description that the LLM sees when deciding whether to call it. Write it clearly; it directly affects call quality.
2. **Type hints** — ADK reads these to generate the JSON schema for parameters. Always annotate every parameter.
3. **Return type** — return a `dict` so ADK can serialize the result cleanly into the conversation context.

Pass the function directly in the `tools=[]` list and ADK handles the rest.

The three tools below are stubs — they return canned data so the lab works without real API calls. In production you would replace their bodies with real HTTP requests or database queries.

In [ ]:
def web_search(query: str) -> dict:
    """Search the web for information on a topic."""
    return {
        "results": [f"Result 1 for {query}", f"Result 2 for {query}"],
        "query": query,
    }


def summarize_text(text: str, max_sentences: int = 3) -> dict:
    """Summarize a block of text into key points."""
    return {
        "summary": f"Summary of: {text[:100]}...",
        "sentence_count": max_sentences,
    }


def fact_check(claim: str) -> dict:
    """Check if a claim appears to be factually accurate."""
    return {"claim": claim, "verdict": "UNVERIFIED", "confidence": 0.5}


print("Tools defined:", web_search.__name__, "/", summarize_text.__name__, "/", fact_check.__name__)

## Section 2 — Building Individual Agents

Each `LlmAgent` wraps a model call with a fixed `instruction` (system prompt) and a set of `tools` it may call. Agents do not share tools automatically — each agent only sees what you give it.

**Key `LlmAgent` parameters:**

| Parameter | Purpose |
|---|---|
| `name` | Unique identifier used in logs and inter-agent references |
| `model` | Model string, e.g. `"gemini-2.5-flash"` |
| `instruction` | System prompt — define the agent's role and constraints here |
| `tools` | List of plain Python functions this agent may invoke |
| `session_service` | Where conversation history is stored (pass the same service to all agents) |

### TODO 1 — Planner Agent

Create `planner_agent`. It should:
- Have access to `web_search`
- Have an instruction that tells it to decompose the research question into 2–3 specific sub-questions and use `web_search` to gather initial context
- Use `session_service` — but we haven't created one yet, so leave that as `None` for now (we'll wire it in Section 4)

In [ ]:
# TODO 1 — Create the planner agent
# Fill in the instruction and pass session_service=None for now.

planner_agent = LlmAgent(
    name="Planner",
    model=MODEL,
    tools=[web_search],
    instruction=None,  # TODO: write a clear instruction here
    session_service=None,
)

# === SOLUTION (reveal only if stuck) ===
# planner_agent = LlmAgent(
#     name="Planner",
#     model=MODEL,
#     tools=[web_search],
#     instruction=(
#         "You are a research planner. Given a research question, decompose it into "
#         "2-3 specific sub-questions that together cover the topic. "
#         "Use web_search to gather initial context on each sub-question. "
#         "Output the sub-questions and search results in a structured list."
#     ),
#     session_service=None,
# )

print("planner_agent created:", planner_agent.name)

### TODO 2 — Parallel Search Agents

Create two `LlmAgent` instances that will run in parallel. Give them different angles on the research — for example, one focuses on technical details while the other focuses on practical comparisons or trade-offs. Both should have access to `web_search`.

In [ ]:
# TODO 2 — Create two parallel search agents with different focus areas

searcher_a = LlmAgent(
    name="SearcherA",
    model=MODEL,
    tools=[web_search],
    instruction=None,  # TODO: focus on technical architecture / internals
    session_service=None,
)

searcher_b = LlmAgent(
    name="SearcherB",
    model=MODEL,
    tools=[web_search],
    instruction=None,  # TODO: focus on practical trade-offs / community feedback
    session_service=None,
)

# === SOLUTION (reveal only if stuck) ===
# searcher_a = LlmAgent(
#     name="SearcherA",
#     model=MODEL,
#     tools=[web_search],
#     instruction=(
#         "You are a technical researcher. Focus on the internal architecture, "
#         "APIs, and design decisions of the frameworks being compared. "
#         "Use web_search to find technical documentation and source analysis."
#     ),
#     session_service=None,
# )
# searcher_b = LlmAgent(
#     name="SearcherB",
#     model=MODEL,
#     tools=[web_search],
#     instruction=(
#         "You are a practical researcher. Focus on real-world developer experience, "
#         "community feedback, and trade-offs reported by practitioners. "
#         "Use web_search to find blog posts, forum discussions, and tutorials."
#     ),
#     session_service=None,
# )

print("searcher_a:", searcher_a.name)
print("searcher_b:", searcher_b.name)

### TODO 3 — Synthesiser Agent

Create a `synthesiser_agent` that receives output from both searchers and combines it into a single coherent answer. It should use `summarize_text` to keep the final output concise.

In [ ]:
# TODO 3 — Create the synthesiser agent

synthesiser_agent = LlmAgent(
    name="Synthesiser",
    model=MODEL,
    tools=[summarize_text],
    instruction=None,  # TODO: synthesise research from all sources into a coherent answer
    session_service=None,
)

# === SOLUTION (reveal only if stuck) ===
# synthesiser_agent = LlmAgent(
#     name="Synthesiser",
#     model=MODEL,
#     tools=[summarize_text],
#     instruction=(
#         "You are a research synthesiser. You receive findings from multiple researchers. "
#         "Use summarize_text to condense each source, then combine them into a single "
#         "well-structured answer that covers both technical and practical perspectives. "
#         "Cite which perspective each point comes from."
#     ),
#     session_service=None,
# )

print("synthesiser_agent:", synthesiser_agent.name)

### TODO 4 — Critic Agent

Create a `critic_agent` that reviews the synthesised answer. It should use `fact_check` on key claims and end its output with either `APPROVED` or `NEEDS_REVISION`.

In [ ]:
# TODO 4 — Create the critic agent

critic_agent = LlmAgent(
    name="Critic",
    model=MODEL,
    tools=[fact_check],
    instruction=None,  # TODO: review the synthesis, fact-check claims, output APPROVED or NEEDS_REVISION
    session_service=None,
)

# === SOLUTION (reveal only if stuck) ===
# critic_agent = LlmAgent(
#     name="Critic",
#     model=MODEL,
#     tools=[fact_check],
#     instruction=(
#         "You are a rigorous fact-checker. Review the synthesised research answer. "
#         "Identify the 2-3 most important factual claims and use fact_check on each. "
#         "Comment on the strength of the evidence. "
#         "End your response with exactly one of: APPROVED or NEEDS_REVISION."
#     ),
#     session_service=None,
# )

print("critic_agent:", critic_agent.name)

## Section 3 — Assembling the Pipeline

ADK gives you two composition primitives:

**`SequentialAgent`** — runs `sub_agents` one after another. Each agent receives the accumulated conversation context, so later agents can see what earlier ones produced.

**`ParallelAgent`** — runs `sub_agents` concurrently. All agents receive the same input context. Their outputs are merged before the next step in a `SequentialAgent` continues.

Nesting rule: you can put a `ParallelAgent` inside a `SequentialAgent`'s sub-agent list — this is how fan-out/fan-in works.

```
SequentialAgent
  [Planner] → [ParallelAgent([SearcherA, SearcherB])] → [Synthesiser]
```

### TODO 5 — Parallel Search Stage

In [ ]:
# TODO 5 — Assemble the parallel search stage
# Use ParallelAgent with searcher_a and searcher_b as sub_agents.

parallel_search = None  # TODO: replace with ParallelAgent(...)

# === SOLUTION (reveal only if stuck) ===
# parallel_search = ParallelAgent(
#     name="ParallelSearch",
#     sub_agents=[searcher_a, searcher_b],
# )

print("parallel_search:", parallel_search)

### TODO 6 — Full Research Pipeline

In [ ]:
# TODO 6 — Assemble the sequential research pipeline
# Order: planner_agent → parallel_search → synthesiser_agent

research_pipeline = None  # TODO: replace with SequentialAgent(...)

# === SOLUTION (reveal only if stuck) ===
# research_pipeline = SequentialAgent(
#     name="ResearchPipeline",
#     sub_agents=[planner_agent, parallel_search, synthesiser_agent],
# )

print("research_pipeline:", research_pipeline)

### Orchestrator — Given

The top-level orchestrator coordinates the full system. It is an `LlmAgent` with `sub_agents` — it can delegate to them as tools. The orchestrator decides when to invoke the research pipeline and when to invoke the critic.

In [ ]:
orchestrator = LlmAgent(
    name="Orchestrator",
    model=MODEL,
    tools=[],
    sub_agents=[research_pipeline, critic_agent],
    instruction=(
        "You coordinate a research system. "
        "First, delegate the user's question to the ResearchPipeline to gather and synthesise information. "
        "Then, delegate the synthesised answer to the Critic for fact-checking and approval. "
        "Present the final approved answer to the user."
    ),
)

print("Orchestrator ready. Sub-agents:", [a.name for a in orchestrator.sub_agents])

## Section 4 — Session Service and Runner

ADK separates **agents** (what to do) from **sessions** (conversation memory) from **runners** (execution engine).

`DatabaseSessionService` persists sessions to a database. With `db_url="sqlite:///./lab_sessions.db"` it writes to a local SQLite file — useful for resuming across notebook restarts.

`InMemorySessionService` is simpler but loses state when the process ends. Good for quick testing.

### TODO 7 — Create a DatabaseSessionService

In [ ]:
# TODO 7 — Create a DatabaseSessionService backed by SQLite
# Use db_url="sqlite:///./lab_sessions.db"

session_service = None  # TODO: replace with DatabaseSessionService(...)

# Hint: if DatabaseSessionService requires an async initialisation step (e.g. create_tables),
# you may need to call it inside an async function below.

# === SOLUTION (reveal only if stuck) ===
# session_service = DatabaseSessionService(db_url="sqlite:///./lab_sessions.db")

print("session_service:", session_service)

### Runner — Given

The `Runner` ties together an agent tree, an app name, and a session service. You call `runner.run_async(...)` with a user ID, session ID, and the new user message. The runner yields events; call `event.is_final_response()` to find the last agent turn.

In [ ]:
async def run_research(question: str):
    """Run the full research pipeline on a question and print the final answer."""
    session = await session_service.create_session(
        app_name="research_lab",
        user_id="learner",
    )
    runner = Runner(
        agent=orchestrator,
        app_name="research_lab",
        session_service=session_service,
    )
    message = Content(parts=[Part(text=question)])
    print(f"Running research on: {question!r}\n")
    async for event in runner.run_async(
        user_id="learner",
        session_id=session.id,
        new_message=message,
    ):
        if event.is_final_response():
            print("=== FINAL ANSWER ===")
            print(event.content.parts[0].text)


# Run it
asyncio.run(
    run_research("What are the main trade-offs between LangGraph and Google ADK?")
)

## Section 4b — Reflection Questions

After running the system, consider:

1. **Parallelism:** `ParallelAgent` runs searchers concurrently. What happens if one searcher fails? Does the pipeline halt or continue?
2. **Context accumulation:** `SequentialAgent` passes accumulated context forward. What is the risk as the pipeline grows longer?
3. **Critic loop:** The critic outputs `APPROVED` or `NEEDS_REVISION` — but our orchestrator does not retry on `NEEDS_REVISION`. How would you add a retry loop?
4. **Session persistence:** Open `lab_sessions.db` with any SQLite viewer. What tables does ADK create? What is stored in each row?

Write your answers in the cell below:

In [ ]:
# Your reflection notes (as Python comments or a multi-line string)

reflections = """
1. Parallelism failure handling:

2. Context accumulation risk:

3. Retry loop design:

4. Session DB schema:
"""

print(reflections)

## Section 5 — Deployment with `adk dev` and Vertex AI

This section is reference only — Vertex AI deployment requires a real GCP project and cannot run inside a notebook.

### Local development with `adk dev`

Once your agent code is in a Python module, run:

```bash
adk dev my_agent_module:orchestrator
```

This starts a local server with a chat UI at `http://localhost:8000`. The UI lets you inspect the full event stream, session state, and tool calls at each step — far faster than print-debugging.

### Vertex AI deployment with `adk_config.yaml`

ADK projects that target Vertex AI use a config file:

```yaml
# adk_config.yaml
project_id: my-gcp-project
location: us-central1
app_name: research_lab
agent_module: my_agent_module
agent_name: orchestrator
session_service:
  type: vertex_datastore          # or sqlite, spanner, firestore
  datastore_id: my-datastore
```

Then deploy with:

```bash
adk deploy --config adk_config.yaml
```

This packages your agent into a Cloud Run service, sets up IAM bindings, and registers it in Vertex AI Agent Builder. The deployed service URL can be called directly or surfaced through Vertex AI's managed agent UI.

### Key differences: local vs Vertex AI

| Aspect | Local (`adk dev`) | Vertex AI (`adk deploy`) |
|---|---|---|
| Session storage | SQLite file | Managed Datastore / Firestore |
| Auth | API key | Service account / Workload Identity |
| Scaling | Single process | Cloud Run auto-scale |
| Cost | Free (model calls only) | Cloud Run + Vertex AI pricing |
| Observability | Terminal logs | Cloud Trace + Cloud Logging |

For production multi-agent systems, Vertex AI also provides **Agent Engine** — a fully managed runtime that handles routing, session management, and observability without you writing any serving code.

## Checkpoint

Before moving on, verify you can:

- [ ] Define a plain Python function as a tool and explain how its docstring affects agent behaviour
- [ ] Compose a `SequentialAgent` containing a `ParallelAgent` as a sub-agent
- [ ] Explain the difference between `InMemorySessionService` and `DatabaseSessionService`
- [ ] Describe what `adk dev` does and when you would use `adk deploy` instead
- [ ] Run the full pipeline on a different research question of your choice

If you want to go further: modify the orchestrator to re-run the research pipeline when the critic outputs `NEEDS_REVISION`. You will need a loop and a way to pass the critic's feedback back into the planner.